In [ ]:
import os
import calendar
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from scipy.spatial import KDTree
from datetime import date

In [ ]:
# ===== USER CONFIGURATION =====

# --- Paths ---
data_path  = os.path.expanduser('../test-data/v3.LR.historical_seaIceRIO_ensembleStats')
mesh_file  = os.path.expanduser('../test-data/mpaso-IcoswISC30E3r5-restart.nc')
route_file = os.path.expanduser('../test-data/arctic_route_sections.nc')
output_dir = os.path.expanduser('../figures')

# --- Vessel types to process ---
# Available: 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7',
#            'IA', 'IAsuper', 'IB', 'IC', 'NIS'
vessel_classes = ['NIS']

# --- Route sections to concatenate, in order ---
# Available sections: A through J (as defined in arctic_route_sections.nc)
route_sections = ['E', 'J', 'C', 'D']
# route_sections = ['H', 'I', 'B', 'A']

# --- Years to include (as strings) ---
year_start = 2020
year_end = 2029
years = [str(y) for y in range(year_start, year_end+1)]

# --- Plot mode ---
# 'transit_time' : transit time in days (NaN = impassable)
# 'RIO'          : minimum RIO along the route (NaN = no data)
# 'passability'  : binary green (passable) / red (impassable)
#plot_mode = 'transit_time'
plot_mode = 'RIO'
# plot_mode = 'passability'

# --- Output ---
save_figs = True   # set False to suppress file saving

# --- RIO heatmap summary method ---
# Controls what value is shown in RIO heatmaps. Passability always uses min.
# 'min'  : route-minimum RIO (worst bottleneck along the route)
# 'mean' : route-mean RIO (average ice severity experienced along the transit)
rio_summary_method = 'min'

# --- x-axis centering ---
# 0-based day-of-year to place at the centre of the heatmap x-axis.
# 258 ~ September 16, near the annual Arctic sea-ice minimum.
plot_center_doy = 258

# --- Season to plot ---
# Selects which half of the (roughly Mar 15 -> Mar 14) season-centered frame
# is shown.  Slicing happens AFTER the x-axis roll, in make_heatmap.
# 'full'   : whole season (default, unchanged behaviour)
# 'spring' : left half only  (~Mar 18 -> mid-Sep)
# 'fall'   : right half only (mid-Sep -> ~Mar 17)
plot_season = 'fall'

# --- Spatial interpolation ---
# Number of nearest MPAS mesh cells to average RIO over at each route waypoint
k_neighbors = 4


In [ ]:
# Speed and impassability configuration per vessel class.
#   'normal_kt'     : service speed in open water / low-risk conditions (knots)
#   'restricted_kt' : speed in the elevated-risk band (PC* only); None = no restricted band
#   'impass_thresh' : RIO below this value -> segment is impassable
VESSEL_CONFIG = {
    'PC1':     {'normal_kt': 19.0,  'restricted_kt': 11.0, 'impass_thresh': -10},
    'PC2':     {'normal_kt': 16.5,  'restricted_kt':  8.0, 'impass_thresh': -10},
    'PC3':     {'normal_kt': 16.0,  'restricted_kt':  5.0, 'impass_thresh': -10},
    'PC4':     {'normal_kt': 15.0,  'restricted_kt':  5.0, 'impass_thresh': -10},
    'PC5':     {'normal_kt': 14.0,  'restricted_kt':  5.0, 'impass_thresh': -10},
    'PC6':     {'normal_kt': 14.0,  'restricted_kt':  3.0, 'impass_thresh': -10},
    'PC7':     {'normal_kt': 14.0,  'restricted_kt':  3.0, 'impass_thresh': -10},
    'IAsuper': {'normal_kt': 14.0,  'restricted_kt': None, 'impass_thresh':   0},
    'IA':      {'normal_kt': 14.0,  'restricted_kt': None, 'impass_thresh':   0},
    'IB':      {'normal_kt': 14.0,  'restricted_kt': None, 'impass_thresh':   0},
    'IC':      {'normal_kt': 14.0,  'restricted_kt': None, 'impass_thresh':   0},
    'NIS':     {'normal_kt': 14.0,  'restricted_kt': None, 'impass_thresh':   0},
}


def rio_to_speed_array(rio_arr, vessel_class):
    """
    Map an array of per-waypoint RIO values to an array of speeds (knots).

    Waypoints that are impassable (RIO < impass_thresh) receive NaN.
    Non-PC vessels have no restricted band: they sail at normal_kt or are
    impassable.

    Parameters
    ----------
    rio_arr      : ndarray, shape (n_waypoints,)
    vessel_class : str   key into VESSEL_CONFIG

    Returns
    -------
    speed : ndarray, shape (n_waypoints,)  in knots; NaN where impassable
    """
    cfg   = VESSEL_CONFIG[vessel_class]
    speed = np.full(len(rio_arr), np.nan, dtype=float)
    thresh = cfg['impass_thresh']

    passable   = (~np.isnan(rio_arr)) & (rio_arr >= thresh)
    normal     = passable & (rio_arr >= 0)
    restricted = passable & (rio_arr < 0)

    speed[normal] = cfg['normal_kt']
    if cfg['restricted_kt'] is not None:
        speed[restricted] = cfg['restricted_kt']
    else:
        # non-PC: no restricted band — sail at normal speed when passable
        speed[restricted] = cfg['normal_kt']

    return speed


def integrate_transit_time(wp_rio, seg_dist_nm, vessel_class):
    """
    Compute segment-integrated transit time in days.

    Each waypoint is assigned a local speed based on its RIO value.
    Transit time = sum(seg_dist_i / speed_i) / 24.
    Returns NaN if any waypoint along the route is impassable.

    Parameters
    ----------
    wp_rio        : ndarray (n_waypoints,)  per-waypoint RIO (k-neighbour average)
    seg_dist_nm   : ndarray (n_waypoints,)  distance represented by each waypoint (NM)
    vessel_class  : str

    Returns
    -------
    float : transit time in days, or NaN if route is impassable
    """
    speed = rio_to_speed_array(wp_rio, vessel_class)
    if np.any(np.isnan(speed)):
        return np.nan
    return float(np.sum(seg_dist_nm / speed) / 24.0)


In [ ]:
print('Loading MPAS mesh...')
ds_mesh = xr.open_dataset(mesh_file)

rad2deg       = 180.0 / np.pi
lat_mesh_full = ds_mesh.latCell.values * rad2deg
lon_mesh_full = ds_mesh.lonCell.values * rad2deg

# Normalise longitudes to [-180, 180]
lon_mesh_full = np.where(lon_mesh_full > 180.0,
                         lon_mesh_full - 360.0,
                         lon_mesh_full)

# Restrict to cells at or north of 60 deg N
ind_north = np.where(lat_mesh_full >= 60.0)[0]
lat_mesh  = lat_mesh_full[ind_north]
lon_mesh  = lon_mesh_full[ind_north]

print(f'  Total mesh cells  : {len(lat_mesh_full):,}')
print(f'  Cells >= 60 deg N : {len(lat_mesh):,}')

In [ ]:
# print('Loading route sections...')
# ds_route = xr.open_dataset(route_file)

# trans_lat_list  = []
# trans_lon_list  = []
# seg_dist_list   = []   # differential distance per waypoint (NM)

# for sec in route_sections:
#     s = sec.upper()
#     dis = ds_route[f'dis_{s}'].values   # cumulative distance (NM), starts at 0
#     trans_lat_list.append(ds_route[f'lat_{s}'].values)
#     trans_lon_list.append(ds_route[f'lon_{s}'].values)
#     # np.diff gives the inter-waypoint spacing; prepend=0 so that the first
#     # waypoint of each section contributes dis[0] NM (which is 0 by convention)
#     seg_dist_list.append(np.diff(dis, prepend=0.0))

# trans_lat    = np.concatenate(trans_lat_list)
# trans_lon    = np.concatenate(trans_lon_list)
# seg_dist_nm  = np.concatenate(seg_dist_list)   # (n_waypoints,)
# total_dist_nm = sum(
#     float(np.max(ds_route[f'dis_{s.upper()}'].values)) for s in route_sections
# )
# route_label  = ''.join(s.upper() for s in route_sections)

# print(f'  Sections        : {route_sections}  ->  {len(trans_lat)} waypoints')
# print(f'  Total distance  : {total_dist_nm:.1f} nm')
# print(f'  seg_dist_nm     : min={seg_dist_nm.min():.3f}, max={seg_dist_nm.max():.3f}, sum={seg_dist_nm.sum():.1f} nm')

# # Build KDTree over the northern-hemisphere restricted mesh
# print('Building KDTree...')
# tree = KDTree(list(zip(lon_mesh, lat_mesh)))
# _, inds = tree.query(list(zip(trans_lon, trans_lat)), k=k_neighbors)
# print(f'  KDTree ready.  Neighbor index array shape: {inds.shape}')


In [ ]:
# ===== USER CONFIGURATION (text-file route) =====
# Path to the plain-text waypoint file.
# Format: one waypoint per line ->  lat_deg, lon_deg, cumulative_dist_nm
# Lines beginning with '#' are treated as comments and ignored.
# 'cumulative_dist_nm' is the odometer distance from the start of the route (NM),
# i.e. each row's value is the total distance travelled so far, not the leg length.
# Example line:  75.1234, 45.1234, 100.0
route_file_txt = os.path.expanduser('../test-data/text_route.txt')
route_label    = os.path.splitext(os.path.basename(route_file_txt))[0]  # or set manually, e.g. 'MYROUTE'

# ---- load waypoints ----
print(f'Loading route from {route_file_txt} ...')

wp_lat_list  = []
wp_lon_list  = []
wp_dist_list = []

with open(route_file_txt) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        parts = [p.strip() for p in line.split(',')]
        if len(parts) < 3:
            continue
        wp_lat_list.append(float(parts[0]))
        wp_lon_list.append(float(parts[1]))
        wp_dist_list.append(float(parts[2]))

trans_lat  = np.array(wp_lat_list)
trans_lon  = np.array(wp_lon_list)
cumul_dist = np.array(wp_dist_list)

# Differential distance per waypoint (NM) — same convention as the netCDF loader.
# np.diff with prepend=0 assigns each waypoint the spacing since the previous one;
# the first waypoint gets cumul_dist[0] (typically 0).
seg_dist_nm   = np.diff(cumul_dist, prepend=0.0)
total_dist_nm = float(np.max(cumul_dist))

print(f'  Waypoints       : {len(trans_lat)}')
print(f'  Total distance  : {total_dist_nm:.1f} nm')
print(f'  seg_dist_nm     : min={seg_dist_nm.min():.3f}, max={seg_dist_nm.max():.3f}, sum={seg_dist_nm.sum():.1f} nm')
print(f'  Route label     : {route_label}')

# Build KDTree over the northern-hemisphere restricted mesh
print('Building KDTree...')
tree = KDTree(list(zip(lon_mesh, lat_mesh)))
_, inds = tree.query(list(zip(trans_lon, trans_lat)), k=k_neighbors)
print(f'  KDTree ready.  Neighbor index array shape: {inds.shape}')

In [ ]:
def _doy_index(year, month, day):
    """Return 0-based day-of-year index (0 = Jan 1)."""
    return (date(year, month, day) - date(year, 1, 1)).days


def _find_files(vessel_class, years, data_path):
    """Return sorted list of (year, month, filepath) tuples for available monthly files."""
    prefix  = (f'v3.LR.historical_{vessel_class}_EnsStats'
               f'.mpassi.hist.am.timeSeriesStatsDaily.')
    records = []
    for yr in years:
        for mo in range(1, 13):
            fname = os.path.join(data_path, f'{prefix}{yr}-{mo:02d}-01.nc')
            if os.path.exists(fname):
                records.append((int(yr), mo, fname))
    return records


# ---- Read and cache raw RIO data (run once) ----
# raw_rio_data[vessel_class] holds a list of (year, month, rio_med_arr, rio_5th_arr, rio_95th_arr)
# where each array has shape (n_days_in_month, n_arctic_cells).
raw_rio_data = {}

for vessel_class in vessel_classes:
    print(f'\n===== {vessel_class} =====')
    file_records = _find_files(vessel_class, years, data_path)

    if not file_records:
        print(f'  WARNING: no files found for {vessel_class} in the specified years -- skipping.')
        continue

    cached_months = []
    for yr, mo, fpath in file_records:
        print(f'  {os.path.basename(fpath)}')
        ds = xr.open_dataset(fpath)

        rio_med_arr  = ds['timeDaily_avg_RIO_ensembleMedian'].values
        rio_5th_arr  = ds['timeDaily_avg_RIO_ensemble5th'].values
        rio_95th_arr = ds['timeDaily_avg_RIO_ensemble95th'].values
        ds.close()

        cached_months.append((yr, mo, rio_med_arr, rio_5th_arr, rio_95th_arr))

    raw_rio_data[vessel_class] = cached_months
    print(f'  Done. {len(cached_months)} file(s) cached.')

print('\nRaw data read complete.')

In [ ]:
# ---- Route-specific extraction (re-run after changing route/waypoints) ----
# Uses: raw_rio_data, inds, seg_dist_nm, rio_summary_method, vessel_classes
results = {}

for vessel_class in vessel_classes:
    if vessel_class not in raw_rio_data:
        print(f'  WARNING: no cached data for {vessel_class} -- skipping.')
        continue

    cached_months = raw_rio_data[vessel_class]
    print(f'\n===== {vessel_class}: extracting route metrics =====')

    avail_years  = sorted(set(yr for yr, _, __, ___, ____ in cached_months))
    n_years      = len(avail_years)
    year_idx_map = {yr: i for i, yr in enumerate(avail_years)}

    shape            = (n_years, 366)
    min_rio_med      = np.full(shape, np.nan)
    min_rio_5th      = np.full(shape, np.nan)
    min_rio_95th     = np.full(shape, np.nan)
    summary_rio_med  = np.full(shape, np.nan)
    summary_rio_5th  = np.full(shape, np.nan)
    summary_rio_95th = np.full(shape, np.nan)
    tt_med           = np.full(shape, np.nan)
    tt_5th           = np.full(shape, np.nan)
    tt_95th          = np.full(shape, np.nan)

    for yr, mo, rio_med_arr, rio_5th_arr, rio_95th_arr in cached_months:
        n_days = rio_med_arr.shape[0]
        yi     = year_idx_map[yr]

        for day_idx in range(n_days):
            doy = _doy_index(yr, mo, day_idx + 1)   # 0-based day-of-year

            # Average k neighbours at each route waypoint: shape (n_waypoints,)
            wp_rio_med  = np.mean(rio_med_arr [day_idx, inds], axis=1)
            wp_rio_5th  = np.mean(rio_5th_arr [day_idx, inds], axis=1)
            wp_rio_95th = np.mean(rio_95th_arr[day_idx, inds], axis=1)

            # Route-minimum RIO (always stored; used for passability)
            min_rio_med [yi, doy] = float(np.nanmin(wp_rio_med))
            min_rio_5th [yi, doy] = float(np.nanmin(wp_rio_5th))
            min_rio_95th[yi, doy] = float(np.nanmin(wp_rio_95th))

            # Summary RIO (method-controlled; used for RIO heatmap)
            if rio_summary_method == 'mean':
                summary_rio_med [yi, doy] = float(np.nanmean(wp_rio_med))
                summary_rio_5th [yi, doy] = float(np.nanmean(wp_rio_5th))
                summary_rio_95th[yi, doy] = float(np.nanmean(wp_rio_95th))
            else:  # 'min'
                summary_rio_med [yi, doy] = float(np.nanmin(wp_rio_med))
                summary_rio_5th [yi, doy] = float(np.nanmin(wp_rio_5th))
                summary_rio_95th[yi, doy] = float(np.nanmin(wp_rio_95th))

            # Segment-integrated transit time
            tt_med [yi, doy] = integrate_transit_time(wp_rio_med,  seg_dist_nm, vessel_class)
            tt_5th [yi, doy] = integrate_transit_time(wp_rio_5th,  seg_dist_nm, vessel_class)
            tt_95th[yi, doy] = integrate_transit_time(wp_rio_95th, seg_dist_nm, vessel_class)

    results[vessel_class] = {
        'years':             avail_years,
        'min_rio_med':       min_rio_med,
        'min_rio_5th':       min_rio_5th,
        'min_rio_95th':      min_rio_95th,
        'summary_rio_med':   summary_rio_med,
        'summary_rio_5th':   summary_rio_5th,
        'summary_rio_95th':  summary_rio_95th,
        'tt_med':            tt_med,
        'tt_5th':            tt_5th,
        'tt_95th':           tt_95th,
    }
    print(f'  Done. {n_years} year(s), {len(cached_months)} file(s) processed.')

print('\nRoute extraction complete.')

In [ ]:
# ---- Export climate-year (Mar 15 -> Mar 14) dataset for trend analysis ----
# Re-indexes each vessel class's RIO / transit-time arrays so that a "season year"
# runs from March 15 of that year through March 14 of the following year
# (DOY 0 = March 15, DOY 364 = March 14 of the following year). This avoids the
# Dec/Jan year-boundary ambiguity entirely for downstream trend detection.
#
# Saved to trendAnalysis/RIO_climate_year_route{route_label}.nc for use by
# trendAnalysis/RIO_trend_climate_year.ipynb. This cell does not modify `results`
# or affect any other cell in this notebook.

_CY_DOY_SHIFT = 73     # 0-based calendar DOY of March 15 (non-leap year)
_CY_N_DAYS    = 365

_cy_dir = os.path.join('.', 'trendAnalysis')
os.makedirs(_cy_dir, exist_ok=True)

_cy_array_keys = [
    'min_rio_med',     'min_rio_5th',     'min_rio_95th',
    'summary_rio_med', 'summary_rio_5th', 'summary_rio_95th',
    'tt_med',          'tt_5th',          'tt_95th',
]

_cy_vessel_classes = list(results.keys())
_cy_split = _CY_N_DAYS - _CY_DOY_SHIFT   # = 292 (Mar15-Dec31 span)

# All vessel classes must share a common season_year axis -- use the shortest
# available year range (in case some vessel classes have fewer cached years).
_cy_season_years = None
for _vc in _cy_vessel_classes:
    _vc_seasons = results[_vc]['years'][:-1]   # last calendar year excluded (no next Jan-Mar)
    _cy_season_years = _vc_seasons if _cy_season_years is None else (
        _cy_season_years if len(_cy_season_years) <= len(_vc_seasons) else _vc_seasons
    )
_cy_n_seasons = len(_cy_season_years)

_cy_data_vars = {}
for _key in _cy_array_keys:
    _cy_stacked = np.full((len(_cy_vessel_classes), _cy_n_seasons, _CY_N_DAYS), np.nan)
    for _vi, _vc in enumerate(_cy_vessel_classes):
        _arr = results[_vc][_key]                      # shape (n_years_raw, 366)
        for _yi in range(_cy_n_seasons):
            _cy_stacked[_vi, _yi, :_cy_split] = _arr[_yi,     _CY_DOY_SHIFT:_CY_N_DAYS]
            _cy_stacked[_vi, _yi, _cy_split:] = _arr[_yi + 1, :_CY_DOY_SHIFT]
    _cy_data_vars[_key] = (('vessel_class', 'season_year', 'doy'), _cy_stacked)

ds_climate_year_rio = xr.Dataset(
    _cy_data_vars,
    coords={
        'vessel_class': _cy_vessel_classes,
        'season_year':  list(_cy_season_years),
        'doy':          np.arange(_CY_N_DAYS),
    },
    attrs={
        'description': ('Climate-year (Mar15-Mar14) re-indexed RIO / transit-time data '
                         'for trend analysis. Row season_year=Y covers March 15 of '
                         'calendar year Y through March 14 of calendar year Y+1.'),
        'doy_convention': ('doy=0 is calendar March 15 of season_year; doy=364 is '
                            'calendar March 14 of season_year+1.'),
        'doy_shift_days':     _CY_DOY_SHIFT,
        'route_label':        route_label,
        'rio_summary_method': rio_summary_method,
        'source_notebook':    'RIO_daily_heatmaps.ipynb',
    },
)

# ---- Validation: spot-check the shift before trusting/saving it ----
_vc0        = _cy_vessel_classes[0]
_orig_years = results[_vc0]['years']
np.testing.assert_allclose(
    ds_climate_year_rio['summary_rio_med'].sel(vessel_class=_vc0, season_year=_orig_years[0], doy=0).values,
    results[_vc0]['summary_rio_med'][0, _CY_DOY_SHIFT],
    err_msg='Climate-year shift mismatch at doy=0 (expected calendar March 15 of first season year).',
)
np.testing.assert_allclose(
    ds_climate_year_rio['summary_rio_med'].sel(vessel_class=_vc0, season_year=_orig_years[0], doy=_cy_split).values,
    results[_vc0]['summary_rio_med'][1, 0],
    err_msg='Climate-year shift mismatch at doy=split (expected calendar January 1 of following year).',
)
print('Climate-year shift spot-checks passed.')

_cy_out_path = os.path.join(_cy_dir, f'RIO_climate_year_route{route_label}.nc')
ds_climate_year_rio.to_netcdf(_cy_out_path)
print(f'Saved climate-year RIO dataset: {_cy_out_path}')
print(f'  vessel_classes = {_cy_vessel_classes}')
print(f'  season_years   = {_cy_season_years[0]}..{_cy_season_years[-1]}  ({_cy_n_seasons} seasons)')


In [ ]:
# Month start positions (0-based day-of-year) and labels for x-axis ticks
_MONTH_DOY   = [0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334]
_MONTH_NAMES = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']


def make_heatmap(data, years, vessel_class, stat_label, plot_mode,
                 route_label, save_figs, output_dir, total_dist_nm=None,
                 center_doy=258, rio_summary_method='min',
                 season='full', trend_overlay=None):
    """
    Produce a single heatmap figure.

    Parameters
    ----------
    data               : ndarray, shape (n_years, 366)
                         transit_time mode : transit times in days (NaN = impassable)
                         RIO / passability : minimum or summary RIO values
    years              : list of int
    vessel_class       : str
    stat_label         : str   e.g. 'Median', '5th percentile'
    plot_mode          : str   'transit_time', 'RIO', or 'passability'
    route_label        : str   e.g. 'HIBA'
    save_figs          : bool
    output_dir         : str
    total_dist_nm      : float  total route length in nautical miles
    center_doy         : int    0-based DOY to place at the centre of the x-axis
    rio_summary_method : str    'min' or 'mean'; used for RIO colorbar label
    season             : str    'full', 'spring', or 'fall'; selects which half of
                                 the season-centered frame to display
    trend_overlay      : dict or None
                         If provided, overlays threshold-crossing scatter points
                         and trend lines on the heatmap.  Expected keys:
                           'spring' and/or 'fall', each a dict with:
                             'yi'        – ndarray of year indices with valid crossings
                             'rolled_x'  – corresponding rolled x-axis positions
                             'slope'     – trend slope (days / year)
                             'intercept' – trend intercept
    """
    n_years  = len(years)
    n_days   = 365         # show days 0-364; omit leap-year slot at index 365
    plot_arr = data[:, :n_days].copy()

    # Roll columns so that center_doy sits in the middle of the x-axis
    start_doy = (center_doy - n_days // 2) % n_days
    plot_arr  = np.roll(plot_arr, -start_doy, axis=1)

    # Optionally keep only one half of the season-centered frame.
    # After the roll, column 0 ~ Mar 18, the centre column (~Sep 16) is n_days//2,
    # and the final column ~ Mar 17.  'spring' = left half, 'fall' = right half.
    split = n_days // 2
    if season == 'spring':
        col0, col1 = 0, split
    elif season == 'fall':
        col0, col1 = split, n_days
    else:  # 'full'
        col0, col1 = 0, n_days
    plot_arr = plot_arr[:, col0:col1]
    n_cols   = plot_arr.shape[1]

    # Month positions in the rolled, sliced frame (offset by col0)
    shifted_starts = [((_MONTH_DOY[m] - start_doy) % n_days) - col0 for m in range(12)]
    month_order    = sorted(range(12), key=lambda m: shifted_starts[m])

    fig_height = max(3.5, n_years * 0.50 + 2.0)
    fig, ax = plt.subplots(figsize=(16, fig_height))

    extent = [-0.5, n_cols - 0.5, -0.5, n_years - 0.5]

    # ------------------------------------------------------------------ #
    # Render according to plot_mode
    # ------------------------------------------------------------------ #
    if plot_mode == 'transit_time':
        cmap = plt.cm.plasma.copy()
        cmap.set_bad(color='lightgrey')
        masked = np.ma.masked_invalid(plot_arr)
        cfg  = VESSEL_CONFIG[vessel_class]
        #vmin = total_dist_nm / cfg['normal_kt'] / 24.0 if total_dist_nm else 0.0
        #vmax = 365.0
        vmin = 12.0
        vmax=15.0
        im = ax.imshow(masked, aspect='auto', origin='lower',
                       cmap=cmap, vmin=vmin, vmax=vmax,
                       extent=extent, interpolation='none')
        cbar = fig.colorbar(im, ax=ax, pad=0.02, fraction=0.03)
        cbar.set_label('Transit time (days)', fontsize=11)

    elif plot_mode == 'RIO':
        # cmap = plt.cm.RdBu_r.copy()
        cmap = plt.cm.RdBu.copy()
        cmap.set_bad(color='lightgrey')
        masked  = np.ma.masked_invalid(plot_arr)
        valid   = masked.compressed()
        abs_max = float(np.nanpercentile(np.abs(valid), 98)) if len(valid) > 0 else 30.0
        # im = ax.imshow(masked, aspect='auto', origin='lower',
        #                cmap=cmap, vmin=-abs_max, vmax=abs_max,
        #                extent=extent, interpolation='none')
        im = ax.imshow(masked, aspect='auto', origin='lower',
                       cmap=cmap, vmin=-30, vmax=30,
                       extent=extent, interpolation='none')        
        cbar = fig.colorbar(im, ax=ax, pad=0.02, fraction=0.03)
        rio_lbl = 'Route-mean RIO' if rio_summary_method == 'mean' else 'Route-minimum RIO (bottleneck)'
        cbar.set_label(rio_lbl, fontsize=11)
        if np.any(~np.isnan(plot_arr)):
            # contour RIO=0 value 
            ax.contour(np.arange(n_cols), np.arange(n_years),
                       np.ma.masked_invalid(plot_arr),
                       levels=[0], colors='black',
                       linewidths=0.8, linestyles='--')
            # contour RIO=-10 value 
            ax.contour(np.arange(n_cols), np.arange(n_years),
                       np.ma.masked_invalid(plot_arr),
                       levels=[-10], colors='black',
                       linewidths=0.8, linestyles='-')

    
    elif plot_mode == 'passability':
        thresh = VESSEL_CONFIG[vessel_class]['impass_thresh']
        binary = np.where(
            (~np.isnan(plot_arr)) & (plot_arr >= thresh), 1.0, 0.0
        )
        cmap_bin = mcolors.ListedColormap(['#d73027', '#1a9850'])
        ax.imshow(binary, aspect='auto', origin='lower',
                  cmap=cmap_bin, vmin=0, vmax=1,
                  extent=extent, interpolation='none')
        green_patch = mpatches.Patch(color='#1a9850', label='Passable')
        red_patch   = mpatches.Patch(color='#d73027', label='Impassable')
        ax.legend(handles=[green_patch, red_patch],
                  loc='upper right', fontsize=10, framealpha=0.9)
    else:
        raise ValueError(f"plot_mode must be 'transit_time', 'RIO', or 'passability'; "
                         f"got '{plot_mode}'")

    # ------------------------------------------------------------------ #
    # Axes formatting
    # ------------------------------------------------------------------ #
    # x-axis: tick marks and labels at the first day of each month.
    # Vertical lines are coincident with the tick marks and label positions.
    # Only keep month markers that fall inside the (possibly sliced) frame.
    tick_positions = []
    tick_labels    = []
    for m in month_order:
        pos = shifted_starts[m]
        if 0 <= pos < n_cols:
            tick_positions.append(pos)
            tick_labels.append(_MONTH_NAMES[m])
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels, fontsize=10)
    ax.set_xlim(-0.5, n_cols - 0.5)

    # Vertical lines at the first day of each month (coincident with ticks)
    for m in range(12):
        pos = shifted_starts[m]
        if 0 < pos < n_cols:
            ax.axvline(x=pos, color='white', linewidth=0.6, alpha=0.5)

    # y-axis: one tick per year, earliest year at the bottom
    ax.set_yticks(range(n_years))
    ax.set_yticklabels([str(y) for y in years], fontsize=10)
    ax.set_ylabel('Year', fontsize=11)
    ax.set_xlabel('Month', fontsize=11)

    mode_titles = {
        'transit_time': 'Transit Time',
        'RIO':          'RIO',
        'passability':  'Go / No-Go',
    }
    ax.set_title(
        f'{vessel_class}  —  {stat_label}  |  '
        f'{mode_titles[plot_mode]}  |  Route {route_label}',
        fontsize=12, fontweight='bold'
    )

    # # ---- Optional trend overlay ---- NOW IN SEPARATE TREND CALCULATION / PLOTTING SUBFOLDER
    # if trend_overlay is not None:
    #     _overlay_styles = {
    #         'spring': {'color': 'limegreen', 'marker': 'o', 'label': 'Spring opening'},
    #         'fall':   {'color': 'orange',    'marker': 's', 'label': 'Fall closing'},
    #     }
    #     legend_handles = []
    #     year_arr = np.array(years, dtype=float)
    #     for season_key, style in _overlay_styles.items():
    #         ov = trend_overlay.get(season_key)
    #         if ov is None:
    #             continue
    #         # Scatter: actual threshold-crossing positions (offset by col0 for slice)
    #         sc = ax.scatter(np.asarray(ov['rolled_x']) - col0, ov['yi'],
    #                         color=style['color'], marker=style['marker'],
    #                         s=40, zorder=5, edgecolors='black', linewidths=0.5,
    #                         label=style['label'])
    #         legend_handles.append(sc)
    #         # Trend line: predicted crossings for all years
    #         if not np.isnan(ov['slope']):
    #             pred_doy    = ov['slope'] * year_arr + ov['intercept']
    #             pred_rolled = np.array([(d - start_doy) % n_days for d in pred_doy]) - col0
    #             trend_yi    = np.arange(n_years, dtype=float)
    #             ax.plot(pred_rolled, trend_yi,
    #                     color=style['color'], linewidth=2, linestyle='--', zorder=4)
    #     if legend_handles:
    #         ax.legend(handles=legend_handles, fontsize=8,
    #                   loc='lower right', framealpha=0.85)

    plt.tight_layout()

    if save_figs:
        os.makedirs(output_dir, exist_ok=True)
        stat_slug = (stat_label.lower()
                     .replace(' ', '_')
                     .replace('th', '')
                     .replace('st', ''))
        overlay_suffix = '_trend' if trend_overlay is not None else ''
        season_suffix  = '' if season == 'full' else f'_{season}'
        fname = os.path.join(
            output_dir,
            f'{vessel_class}_route{route_label}_{plot_mode}_{stat_slug}{season_suffix}{overlay_suffix}.png'
        )
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        print(f'  Saved: {fname}')

    plt.show()
    plt.close(fig)


In [ ]:
for vessel_class, res in results.items():
    print(f'\nPlotting {vessel_class}  (mode: {plot_mode}) ...')

    if plot_mode == 'transit_time':
        # Transit-time arrays; NaN where impassable
        data_trio = [
            (res['tt_med'],           'Median'),
            (res['tt_5th'],           '5th percentile'),
            (res['tt_95th'],          '95th percentile'),
        ]
    elif plot_mode == 'passability':
        # Passability always uses the route-minimum RIO regardless of rio_summary_method
        data_trio = [
            (res['min_rio_med'],      'Median'),
            (res['min_rio_5th'],      '5th percentile'),
            (res['min_rio_95th'],     '95th percentile'),
        ]
    elif plot_mode == 'RIO':
        # RIO heatmap uses whichever summary method is selected in CONFIG
        data_trio = [
            (res['summary_rio_med'],  'Median'),
            (res['summary_rio_5th'],  '5th percentile'),
            (res['summary_rio_95th'], '95th percentile'),
        ]
    else:
        raise ValueError(f"Unknown plot_mode: '{plot_mode}'")

    for data, stat_label in data_trio:
        make_heatmap(
            data               = data,
            years              = res['years'],
            vessel_class       = vessel_class,
            stat_label         = stat_label,
            plot_mode          = plot_mode,
            route_label        = route_label,
            save_figs          = save_figs,
            output_dir         = output_dir,
            total_dist_nm      = total_dist_nm,
            center_doy         = plot_center_doy,
            rio_summary_method = rio_summary_method,
            season             = plot_season,
        )
